# Global Mortality Dashboard Analysis (1970-2010)
## Interactive analysis of age-specific mortality patterns with predictive forecasting

## 1. Data Loading and Preparation

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load and prepare data
df = pd.read_csv('IHME_GBD_2010_MORTALITY_AGE_SPECIFIC_BY_COUNTRY_1970_2010.csv')

# Clean column names and data
df.columns = df.columns.str.strip()
df['Number of Deaths'] = df['Number of Deaths'].str.replace(',', '').astype(int)
df['Death Rate Per 100,000'] = df['Death Rate Per 100,000'].str.replace(',', '').astype(float)

print(f"Dataset shape: {df.shape}")
print(f"Countries: {df['Country Name'].nunique()}")
print(f"Years: {df['Year'].min()} - {df['Year'].max()}")
df.head()

Dataset shape: (58905, 7)
Countries: 187
Years: 1970 - 2010


,Country Code,Country Name,Year,Age Group,Sex,Number of Deaths,"Death Rate Per 100,000"
0,AFG,Afghanistan,1970,0-6 days,Male,19241,318292.9
1,AFG,Afghanistan,1970,0-6 days,Female,12600,219544.2
2,AFG,Afghanistan,1970,0-6 days,Both,31840,270200.7
3,AFG,Afghanistan,1970,7-27 days,Male,15939,92701.0
4,AFG,Afghanistan,1970,7-27 days,Female,11287,68594.5


In [18]:
# Add region mapping
region_mapping = {
    'AFG': 'South Asia', 'AGO': 'Sub-Saharan Africa', 'ALB': 'Europe & Central Asia',
    'USA': 'North America', 'CHN': 'East Asia & Pacific', 'IND': 'South Asia', 
    'BRA': 'Latin America & Caribbean', 'RUS': 'Europe & Central Asia',
    'JPN': 'East Asia & Pacific', 'DEU': 'Europe & Central Asia', 'GBR': 'Europe & Central Asia',
    'FRA': 'Europe & Central Asia', 'ITA': 'Europe & Central Asia', 'CAN': 'North America',
    'KOR': 'East Asia & Pacific', 'ESP': 'Europe & Central Asia', 'MEX': 'Latin America & Caribbean',
    'IDN': 'East Asia & Pacific', 'TUR': 'Europe & Central Asia', 'SAU': 'Middle East & North Africa',
    'NLD': 'Europe & Central Asia', 'CHE': 'Europe & Central Asia'
}

df['Region'] = df['Country Code'].map(region_mapping).fillna('Other')
print(f"Regions: {df['Region'].unique()}")

Regions: ['South Asia' 'Sub-Saharan Africa' 'Europe & Central Asia' 'Other'
 'Latin America & Caribbean' 'North America' 'East Asia & Pacific'
 'Middle East & North Africa']


## 2. Exploratory Data Analysis

In [19]:
# Key Performance Indicators
countries = ['United States', 'China', 'India', 'Germany', 'Japan']
filtered_df = df[
    (df['Country Name'].isin(countries)) &
    (df['Sex'] == 'Both') &
    (df['Age Group'] == 'All ages')
]

total_deaths = filtered_df['Number of Deaths'].sum()
avg_death_rate = filtered_df['Death Rate Per 100,000'].mean()
countries_count = filtered_df['Country Name'].nunique()

print(f"Total Deaths: {total_deaths:,}")
print(f"Average Death Rate per 100K: {avg_death_rate:.1f}")
print(f"Countries Analyzed: {countries_count}")

Total Deaths: 106,555,194
Average Death Rate per 100K: 930.5
Countries Analyzed: 5


In [20]:
# Mortality Rate Trends Over Time
fig = px.line(filtered_df, x='Year', y='Death Rate Per 100,000', 
              color='Country Name', title='Mortality Rate Trends Over Time (1970-2010)',
              labels={'Death Rate Per 100,000': 'Death Rate per 100,000'})
fig.update_layout(height=500)
fig.show()

In [21]:
# Regional Comparison
regional_data = df[
    (df['Sex'] == 'Both') & 
    (df['Age Group'] == 'All ages')
].groupby('Region')['Death Rate Per 100,000'].mean().reset_index()

fig = px.bar(regional_data, x='Region', y='Death Rate Per 100,000',
             title='Average Mortality Rate by Region',
             labels={'Death Rate Per 100,000': 'Avg Death Rate per 100,000'})
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [22]:
# Age Distribution Analysis
age_data = df[
    (df['Country Name'].isin(countries)) &
    (df['Sex'] == 'Both') &
    (df['Age Group'] != 'All ages')
].groupby('Age Group')['Death Rate Per 100,000'].mean().reset_index()

fig = px.bar(age_data, x='Age Group', y='Death Rate Per 100,000',
             title='Mortality Rate by Age Group',
             labels={'Death Rate Per 100,000': 'Avg Death Rate per 100,000'})
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [23]:
# Mortality Heatmap
heatmap_data = df[
    (df['Country Name'].isin(countries)) &
    (df['Sex'] == 'Both') &
    (df['Age Group'] != 'All ages')
].pivot_table(
    values='Death Rate Per 100,000', 
    index='Age Group', 
    columns='Country Name', 
    aggfunc='mean'
)

fig = px.imshow(heatmap_data, 
                title='Mortality Rate Heatmap: Age Groups vs Countries',
                labels={'color': 'Death Rate per 100,000'},
                aspect='auto')
fig.update_layout(height=500)
fig.show()

## 3. Advanced EDA Components

In [24]:
# Top 10 Countries by Mortality Rate
top_countries = df[
    (df['Sex'] == 'Both') & 
    (df['Age Group'] == 'All ages')
].groupby('Country Name')['Death Rate Per 100,000'].mean().nlargest(10).reset_index()

fig = px.bar(top_countries, x='Death Rate Per 100,000', y='Country Name',
             title='Top 10 Countries by Mortality Rate',
             orientation='h', color='Death Rate Per 100,000',
             color_continuous_scale='Reds')
fig.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig.show()

In [25]:
# Mortality Rate Distribution
fig = px.histogram(filtered_df, x='Death Rate Per 100,000',
                   title='Mortality Rate Distribution', nbins=20,
                   color_discrete_sequence=['#3498db'])
fig.update_layout(height=400)
fig.show()

In [26]:
# Sex Comparison Over Time
sex_comparison = df[
    (df['Country Name'].isin(countries)) &
    (df['Age Group'] == 'All ages') &
    (df['Sex'].isin(['Male', 'Female']))
].groupby(['Year', 'Sex'])['Death Rate Per 100,000'].mean().reset_index()

fig = px.line(sex_comparison, x='Year', y='Death Rate Per 100,000',
              color='Sex', title='Mortality Rate by Sex Over Time',
              color_discrete_map={'Male': '#3498db', 'Female': '#e74c3c'})
fig.update_layout(height=400)
fig.show()

In [27]:
# Correlation Analysis
fig = px.scatter(filtered_df, x='Number of Deaths', y='Death Rate Per 100,000',
                 color='Country Name', size='Year',
                 title='Deaths vs Death Rate Correlation',
                 hover_data=['Year', 'Age Group'])
fig.update_layout(height=500)
fig.show()

In [35]:
# Global Mortality Map
map_data = df[
    (df['Sex'] == 'Both') & 
    (df['Age Group'] == 'All ages')
].groupby(['Country Code', 'Country Name'])['Death Rate Per 100,000'].mean().reset_index()

fig = px.choropleth(
    map_data,
    locations='Country Code',
    color='Death Rate Per 100,000',
    hover_name='Country Name',
    hover_data={'Country Code': False, 'Death Rate Per 100,000': ':.1f'},
    color_continuous_scale='Reds',
    title='Global Mortality Rates (1970-2010) - Average Death Rate per 100,000',
    labels={'Death Rate Per 100,000': 'Death Rate per 100,000'}
)

fig.update_layout(
    height=600,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    )
)

fig.show()

## 4. Forecasting Analysis

In [33]:
def create_forecast(data, years_ahead, model_type='linear'):
    """Create mortality rate forecast using specified model"""
    if len(data) < 3:
        return None, None
    
    X = data['Year'].values.reshape(-1, 1)
    y = data['Death Rate Per 100,000'].values
    
    if model_type == 'linear':
        model = LinearRegression()
    elif model_type == 'poly2':
        model = Pipeline([('poly', PolynomialFeatures(degree=2)), ('linear', LinearRegression())])
    elif model_type == 'poly3':
        model = Pipeline([('poly', PolynomialFeatures(degree=3)), ('linear', LinearRegression())])
    
    model.fit(X, y)
    
    last_year = data['Year'].max()
    future_years = np.arange(last_year + 1, last_year + years_ahead + 1).reshape(-1, 1)
    predictions = model.predict(future_years)
    predictions = np.maximum(predictions, 0)
    
    return future_years.flatten(), predictions

In [34]:
# Mortality Rate Forecasting
fig = go.Figure()

# Define consistent colors for countries
country_colors = px.colors.qualitative.Set1[:len(countries)]
country_color_map = {country: country_colors[i] for i, country in enumerate(countries)}

forecast_periods = [10, 20, 30]
model_type = 'linear'

for country in countries:
    country_data = filtered_df[filtered_df['Country Name'] == country].sort_values('Year')
    
    if len(country_data) < 2:
        continue
    
    base_color = country_color_map[country]
    
    # Historical data
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['Death Rate Per 100,000'],
        mode='lines+markers',
        name=f'{country} - Historical',
        line=dict(width=3, color=base_color),
        marker=dict(size=6, color=base_color)
    ))
    
    # Forecasts
    for period in forecast_periods:
        future_years, predictions = create_forecast(country_data, period, model_type)
        
        if future_years is not None:
            fig.add_trace(go.Scatter(
                x=future_years,
                y=predictions,
                mode='lines',
                name=f'{country} - {period}Y Forecast',
                line=dict(dash='dash', width=2.5, color=base_color),
                opacity=0.7
            ))

# Add vertical line to separate historical and forecast
last_historical_year = df['Year'].max()
fig.add_vline(x=last_historical_year, line_dash="dot", line_color="gray", 
              annotation_text="Historical | Forecast", annotation_position="top")

fig.update_layout(
    title=f'Mortality Rate Forecasting: 10-30 Years Ahead ({model_type.title()} Model)',
    xaxis_title='Year',
    yaxis_title='Death Rate per 100,000',
    height=600,
    hovermode='x unified'
)

fig.show()

## 5. Key Insights and Analysis

In [30]:
# Calculate trend analysis
first_year_data = filtered_df[filtered_df['Year'] == filtered_df['Year'].min()]
last_year_data = filtered_df[filtered_df['Year'] == filtered_df['Year'].max()]

first_year_rate = first_year_data['Death Rate Per 100,000'].mean()
last_year_rate = last_year_data['Death Rate Per 100,000'].mean()
overall_change = ((last_year_rate - first_year_rate) / first_year_rate) * 100

print(f"Overall mortality rate change (1970-2010): {overall_change:+.1f}%")
print(f"First year average rate: {first_year_rate:.1f}")
print(f"Last year average rate: {last_year_rate:.1f}")

Overall mortality rate change (1970-2010): -19.9%
First year average rate: 1086.8
Last year average rate: 870.7


In [31]:
# Country performance analysis
country_trends = []
for country in countries:
    country_first = first_year_data[first_year_data['Country Name'] == country]['Death Rate Per 100,000'].values
    country_last = last_year_data[last_year_data['Country Name'] == country]['Death Rate Per 100,000'].values
    
    if len(country_first) > 0 and len(country_last) > 0:
        change = ((country_last[0] - country_first[0]) / country_first[0]) * 100
        country_trends.append((country, change, country_first[0], country_last[0]))

country_trends.sort(key=lambda x: x[1])

print("\nCountry Performance (1970-2010):")
for country, change, first_rate, last_rate in country_trends:
    print(f"{country}: {change:+.1f}% ({first_rate:.1f} → {last_rate:.1f})")


Country Performance (1970-2010):
India: -50.9% (1652.3 → 811.2)
China: -33.1% (924.6 → 618.8)
Germany: -14.9% (1226.9 → 1043.8)
United States: -8.5% (949.6 → 869.2)
Japan: +48.5% (680.4 → 1010.3)


## 6. Summary Statistics

In [32]:
# Summary statistics
print("Dataset Summary:")
print(f"Total records: {len(df):,}")
print(f"Countries: {df['Country Name'].nunique()}")
print(f"Years covered: {df['Year'].min()} - {df['Year'].max()}")
print(f"Age groups: {df['Age Group'].nunique()}")
print(f"Sex categories: {df['Sex'].unique()}")
print(f"Regions: {df['Region'].nunique()}")

print("\nMortality Rate Statistics:")
print(df['Death Rate Per 100,000'].describe())

Dataset Summary:
Total records: 58,905
Countries: 187
Years covered: 1970 - 2010
Age groups: 21
Sex categories: ['Male' 'Female' 'Both']
Regions: 8

Mortality Rate Statistics:
count     58905.000000
mean       7062.866458
std       24582.548947
min           5.500000
25%         210.300000
50%         825.000000
75%        3611.800000
max      423790.200000
Name: Death Rate Per 100,000, dtype: float64
